# Lab 5 part 1
## Task 1

In [73]:
import pandas as pd

rentals_raw = pd.read_csv("../data/bike_rentals.csv")
rentals = rentals_raw.copy()

print(rentals.shape)
print(rentals.dtypes)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object


In [74]:
column_empty_status = rentals.isna().all()
empty_columns_after = rentals.columns[column_empty_status]
print(column_empty_status)
print("Empty columns", empty_columns_after)

rental_id        False
station          False
bike_type        False
helmets_text     False
is_member        False
insured          False
paid_online      False
status           False
coupon_code       True
damage_notes      True
late_fee_note     True
start_date       False
end_date         False
price            False
rating           False
dtype: bool
Empty columns Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')


In [75]:
rentals = rentals.dropna(axis="columns", how="all")
print(rentals.shape)
column_empty_status = rentals.isna().all()
empty_columns_after = rentals.columns[column_empty_status]
print(column_empty_status)
print("Empty columns after", empty_columns_after)

(403, 12)
rental_id       False
station         False
bike_type       False
helmets_text    False
is_member       False
insured         False
paid_online     False
status          False
start_date      False
end_date        False
price           False
rating          False
dtype: bool
Empty columns after Index([], dtype='str')


## Task 2

In [76]:
rentals["price"].head(3)

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
Name: price, dtype: str

In [77]:
# Step 1: remove the currency text and any remaning spaces
rentals["price"] = rentals["price"].str.replace("SEK", "").str.strip()
rentals["price"].head(3)

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str

In [78]:
# Step 2: remove the comma
rentals["price"] = rentals["price"].str.replace(",", "")
rentals["price"].head(3)

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

In [79]:
rentals["price"] = rentals["price"].astype(float)
print(rentals["price"].dtype)
print(rentals["price"].describe())

float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64


In [80]:
top_3_price = rentals.sort_values("price", ascending=False)[["rental_id", "bike_type", "status", "price"]].head(3)
top_3_price

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


## Task 3

In [81]:
missing_prices = rentals[rentals["price"].isna()]
print("Missing prices count", missing_prices.shape[0])
print("Missing price percentage", round(missing_prices.shape[0] / rentals.shape[0] * 100, 1))

Missing prices count 51
Missing price percentage 12.7


In [82]:
missing = rentals["price"].isna()
cancelled = rentals["status"] == "Cancelled"
pd.crosstab(missing, cancelled, rownames=["Price is missing"], colnames=["Rental cancelled"])

Rental cancelled,False,True
Price is missing,,
False,352,0
True,4,47


47 of the missing prices occur when the rental was cancelled, so they have a reason to be missing.  
The remaining 4 missing prices are missing due to some other reason.  

I will choose to keep the missing prices as they are, and not remove them or fill them in.  

## Task 4

In [83]:
tf_columns = [col for col in rentals.columns if set(rentals[col].dropna().unique()).issubset({"t", "f"})]
tf_columns

['is_member', 'insured', 'paid_online']

The column `status` is not in the list of columns, since its values are not only `t` or `f`.

In [84]:
for column in tf_columns:
    rentals[column] = rentals[column].map({"t": True, "f": False})

In [85]:
rentals.head()

,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating
0,R-5020,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-04-27,2026-04-29,855.5,5.0
1,R-5100,Södermalm,E-bike,1 helmet,False,True,True,Completed,2026-09-16,2026-09-19,1311.5,4.0
2,R-5055,Södermalm,City bike,3 helmets,False,False,True,Completed,2026-08-04,2026-08-06,350.0,NaN
3,R-5200,Södermalm,City bike,2 helmets,False,False,True,No-show,2026-05-30,2026-06-02,569.5,NaN
4,R-5109,Östermalm,City bike,2 helmets,False,True,False,No-show,2026-04-25,2026-04-29,828.0,NaN


In [86]:
for column in tf_columns:
    print(f"{column} has {rentals[column].isna().sum()} missing values")

is_member has 0 missing values
insured has 3 missing values
paid_online has 0 missing values


In [90]:
rentals.groupby("is_member")["price"].agg(["median", "count"])

,median,count
is_member,,
False,756.25,248
True,558.00,104
